In [8]:
import numpy as np
import torch
import time
from scipy import sparse as sp
from scipy.linalg import eigh
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

pauli_matrices = {
    1: np.array([[0, 1], [1, 0]], dtype=complex),
    2: np.array([[0, -1j], [1j, 0]], dtype=complex),
    3: np.array([[1, 0], [0, -1]], dtype=complex),
    0: np.eye(2, dtype=complex)
}
pauli_matrices_sparse = {key: sp.csr_matrix(value) for key, value in pauli_matrices.items()}

def pauli_site(x, i, L):
    if L == 1:
        return pauli_matrices_sparse[x]
    if i == 1:
        return sp.kron(pauli_matrices_sparse[x], sp.eye(2**(L-1), format='csr'))
    else:
        return sp.kron(sp.eye(2), pauli_site(x, i-1, L-1), format='csr')

def pauli_int(spin, site, L):
    if site == L:
        op = sp.eye(2**L, dtype=complex, format='csr')
        for i in range(1, L+1):
            if i == L or i == 1:
                op = op @ pauli_site(spin, i, L)
        return op
    else:
        op = sp.eye(2**L, dtype=complex, format='csr')
        for i in range(1, L+1):
            if i == site or i == site+1:
                op = op @ pauli_site(spin, i, L)
        return op

def xxz_hamiltonian(L, delta=1):
    dim = 2**L
    H = sp.csr_matrix((dim, dim), dtype=complex)
    for site in range(1, L+1):
        H += pauli_int(1, site, L)
        H += pauli_int(2, site, L)
        H += delta * pauli_int(3, site, L)
    return H

def next_nearest_neighbor_interaction(L):
    dim = 2**L
    H1 = sp.csr_matrix((dim, dim), dtype=complex)
    for n in range(1, L+1):
        n2 = n + 2
        if n2 > L:
            n2 = n2 - L
        H1 += pauli_site(1, n, L) @ pauli_site(1, n2, L)
        H1 += pauli_site(2, n, L) @ pauli_site(2, n2, L)
        H1 += pauli_site(3, n, L) @ pauli_site(3, n2, L)
    return H1

def build_hamiltonian(model, L, **kwargs):
    if model == "NextNearest":
        delta = kwargs.get('delta', 1.0)
        lambda_val = kwargs.get('lambda_val', 0.3)
        H0 = xxz_hamiltonian(L, delta)
        H1 = next_nearest_neighbor_interaction(L)
        return H0 + lambda_val * H1

def local_ops(dtype=torch.complex64, device="cpu"):
    Sz = torch.tensor([[0.5, 0.0], [0.0, -0.5]], dtype=dtype, device=device)
    Sp = torch.tensor([[0.0, 1.0], [0.0, 0.0]], dtype=dtype, device=device)
    Sm = torch.tensor([[0.0, 0.0], [1.0, 0.0]], dtype=dtype, device=device)
    Sx = torch.tensor([[0.0, 1.0], [1.0, 0.0]], dtype=dtype, device=device)
    Sy = torch.tensor([[0.0, -1j], [1j, 0.0]], dtype=dtype, device=device)
    I2 = torch.eye(2, dtype=dtype, device=device)
    return I2, Sx, Sy, Sz, Sp, Sm

def precompute_perms(L: int):
    perms, inv_perms = [], []
    axes = list(range(L))
    for s in range(L):
        p = [s] + [a for a in axes if a != s]
        inv = np.argsort(p)
        perms.append(p)
        inv_perms.append(inv.tolist())
    return perms, inv_perms

def apply_one_site(op2, site, vec, L, perms, inv_perms):
    psi = vec.reshape([2]*L).permute(perms[site]).reshape(2, -1)
    psi2 = op2 @ psi
    out = psi2.reshape([2]*L).permute(inv_perms[site]).reshape(-1)
    return out

def B_action(u, psi, L, I2, Sz, Sp, Sm, perms, inv_perms):
    dtype, device = psi.dtype, psi.device
    ii = torch.tensor(1j, dtype=dtype, device=device)
    V11 = psi.clone()
    V12 = torch.zeros_like(psi)
    V21 = torch.zeros_like(psi)
    V22 = psi.clone()
    for site in range(L):
        SzV11 = apply_one_site(Sz, site, V11, L, perms, inv_perms)
        SzV12 = apply_one_site(Sz, site, V12, L, perms, inv_perms)
        SzV21 = apply_one_site(Sz, site, V21, L, perms, inv_perms)
        SzV22 = apply_one_site(Sz, site, V22, L, perms, inv_perms)
        SmV21 = apply_one_site(Sm, site, V21, L, perms, inv_perms)
        SmV22 = apply_one_site(Sm, site, V22, L, perms, inv_perms)
        SpV11 = apply_one_site(Sp, site, V11, L, perms, inv_perms)
        SpV12 = apply_one_site(Sp, site, V12, L, perms, inv_perms)
        L11V11 = u * V11 + ii * SzV11
        L11V12 = u * V12 + ii * SzV12
        L22V21 = u * V21 - ii * SzV21
        L22V22 = u * V22 - ii * SzV22
        L12V21 = ii * SmV21
        L12V22 = ii * SmV22
        L21V11 = ii * SpV11
        L21V12 = ii * SpV12
        N11 = L11V11 + L12V21
        N12 = L11V12 + L12V22
        N21 = L21V11 + L22V21
        N22 = L21V12 + L22V22
        V11, V12, V21, V22 = N11, N12, N21, N22
    return V12

def vacuum_state(L, dtype=torch.complex64, device="cpu"):
    v = torch.zeros(2**L, dtype=dtype, device=device)
    v[0] = 1.0 + 0.0j
    return v

def bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True):
    psi = vacuum_state(L, dtype=I2.dtype, device=I2.device)
    for layer_params in params_list:
        u = layer_params[0] + 1j * layer_params[1]
        psi = B_action(u, psi, L, I2, Sz, Sp, Sm, perms, inv_perms)
    if normalize:
        norm = torch.sqrt(torch.real(torch.vdot(psi, psi)))
        if norm > 1e-15:
            psi = psi / norm
    return psi

def exact_diagonalization(H, num_states=10):
    if sp.issparse(H):
        H_dense = H.toarray()
    else:
        H_dense = H
    eigenvalues, eigenvectors = eigh(H_dense)
    results = []
    for i in range(min(num_states, len(eigenvalues))):
        results.append({'energy': eigenvalues[i], 'state': eigenvectors[:, i]})
    return results

def find_degenerate_groups(exact_states, tol=1e-10):
    groups = []
    current_group = [0]
    for i in range(1, len(exact_states)):
        if abs(exact_states[i]['energy'] - exact_states[i-1]['energy']) < tol:
            current_group.append(i)
        else:
            groups.append(current_group)
            current_group = [i]
    groups.append(current_group)
    return groups

def pytorch_energy_loss(params, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers):
    params_per_layer = 2
    total_layers = len(params) // params_per_layer
    params_list = []
    for layer in range(total_layers):
        start_idx = layer * params_per_layer
        re_part = params[start_idx]
        im_part = params[start_idx + 1]
        params_list.append([re_part, im_part])
    psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)
    H_psi = H_tensor @ psi
    energy = torch.real(torch.vdot(psi, H_psi))
    norm2 = torch.real(torch.vdot(psi, psi))
    return energy / norm2

def torch_wrapper_loss(params_numpy, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers):
    params = torch.tensor(params_numpy, dtype=torch.float64, requires_grad=True)
    loss = pytorch_energy_loss(params, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers)
    loss.backward()
    return loss.item(), params.grad.numpy().astype(np.float64)

def pytorch_penalty_loss(params, H_tensor, lower_states_tensor, L, I2, Sz, Sp, Sm,
                         perms, inv_perms, num_layers, penalty_weight):
    params_per_layer = 2
    total_layers = len(params) // params_per_layer
    params_list = []
    for layer in range(total_layers):
        start_idx = layer * params_per_layer
        re_part = params[start_idx]
        im_part = params[start_idx + 1]
        params_list.append([re_part, im_part])
    psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)
    H_psi = H_tensor @ psi
    energy = torch.real(torch.vdot(psi, H_psi))
    norm2 = torch.real(torch.vdot(psi, psi))
    energy_expect = energy / norm2
    penalty = 0.0
    for phi in lower_states_tensor:
        overlap = torch.abs(torch.vdot(psi, phi)) ** 2 / norm2
        penalty = penalty + overlap
    total_loss = energy_expect + penalty_weight * penalty
    return total_loss

def torch_wrapper_penalty(params_numpy, H_tensor, lower_states_tensor, L, I2, Sz, Sp, Sm,
                          perms, inv_perms, num_layers, penalty_weight):
    params = torch.tensor(params_numpy, dtype=torch.float64, requires_grad=True)
    loss = pytorch_penalty_loss(params, H_tensor, lower_states_tensor, L, I2, Sz, Sp, Sm,
                                perms, inv_perms, num_layers, penalty_weight)
    loss.backward()
    return loss.item(), params.grad.numpy().astype(np.float64)

def fit_ground_state_bethe(H, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms,
                           num_layers, num_attempts, maxiter, device):
    """Fit the ground state using Bethe ansatz (energy minimization). Returns fitted wavefunction (torch tensor)."""
    best_energy = float('inf')
    best_psi = None
    for attempt in range(num_attempts):
        total_params = num_layers * 2
        x0 = np.zeros(total_params, dtype=np.float64)
        for layer in range(num_layers):
            x0[layer*2:layer*2+2] = np.random.normal(-1, 1, 2)
        def loss_wrapper(p):
            return torch_wrapper_loss(p, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers)
        try:
            result = minimize(
                lambda p: loss_wrapper(p)[0],
                x0,
                method='L-BFGS-B',
                jac=lambda p: loss_wrapper(p)[1],
                options={'maxiter': maxiter, 'ftol': 1e-12, 'gtol': 1e-10},
                bounds=[(-3, 3)] * total_params
            )
            final_params = result.x
            params_list = []
            for layer in range(num_layers):
                re_part = final_params[layer*2]
                im_part = final_params[layer*2+1]
                params_list.append([re_part, im_part])
            psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)
            psi_numpy = psi.cpu().detach().numpy()
            if sp.issparse(H):
                H_psi = H.dot(psi_numpy)
            else:
                H_psi = H @ psi_numpy
            energy = np.real(np.vdot(psi_numpy, H_psi) / np.vdot(psi_numpy, psi_numpy))
            if energy < best_energy:
                best_energy = energy
                best_psi = psi  # keep as tensor
        except:
            continue
    if best_psi is None:
        raise RuntimeError("Ground state fitting failed.")
    print(f"  Fitted ground state energy = {best_energy:.8f}")
    return best_psi.detach().clone()

def optimize_subspace(model, L, target_level=0, penalty_weight=10.0, num_layers=3,
                      num_attempts=5, maxiter=6000, device="cpu",
                      use_fitted_ground=False,  
                      **model_kwargs):
    """
    Optimize a Bethe state to approximate the subspace of a given energy level.

    Parameters:
        target_level : int
            0 = ground state, 1 = first excited group, 2 = second excited group, etc.
        penalty_weight : float
            Weight for orthogonality penalty to lower subspaces (ignored for target_level=0).
        use_fitted_ground : bool
            If True and target_level == 1 and ground state is non-degenerate, the penalty will use
            a fitted Bethe ground state (instead of exact eigenstates). This allows avoiding
            exact diagonalization for the penalty term, useful for large systems where exact
            diagonalization is not feasible. For degenerate ground states or higher target levels,
            the code automatically falls back to using exact eigenstates.
    """
    print(f"\n===== Subspace optimization =====\nModel: {model}, L={L}, target level = {target_level}")
    print(f"Parameters: {model_kwargs}")

    H = build_hamiltonian(model, L, **model_kwargs)
    exact_states = exact_diagonalization(H, num_states=20)
    groups = find_degenerate_groups(exact_states)
    print(f"Degeneracy groups (energy, multiplicity):")
    for idx, grp in enumerate(groups):
        e = exact_states[grp[0]]['energy']
        print(f"  Group {idx}: energy = {e:.8f}, multiplicity = {len(grp)}")
    
    if target_level >= len(groups):
        raise ValueError(f"Target level {target_level} not available; only {len(groups)} groups found.")
    
    target_indices = groups[target_level]
    lower_indices = []
    for g in groups[:target_level]:
        lower_indices.extend(g)
    
    print(f"Target subspace indices: {target_indices}")
    if lower_indices:
        print(f"Lower subspace indices: {lower_indices}")

    I2, Sx, Sy, Sz, Sp, Sm = local_ops(dtype=torch.complex128, device=device)
    perms, inv_perms = precompute_perms(L)
    if sp.issparse(H):
        H_dense = H.toarray()
    else:
        H_dense = H
    H_tensor = torch.tensor(H_dense, dtype=torch.complex128, device=device)
    
    lower_states_tensor = []
    use_fitted = False
    if target_level > 0:
        if use_fitted_ground and target_level == 1 and len(groups[0]) == 1:
            try:
                fitted_ground = fit_ground_state_bethe(
                    H, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms,
                    num_layers=num_layers, num_attempts=num_attempts,
                    maxiter=maxiter, device=device
                )
                lower_states_tensor = [fitted_ground]
                use_fitted = True
            except Exception as e:
                print(f"  Fitting ground state failed: {e}. Falling back to exact eigenstates.")
                use_fitted = False
        if not use_fitted:
            for idx in lower_indices:
                lower_states_tensor.append(torch.tensor(exact_states[idx]['state'], dtype=torch.complex128, device=device))
    else:
        pass
    
    best_energy = float('inf')
    best_params = None
    best_psi = None
    best_fidelity = 0.0
    
    for attempt in range(num_attempts):
        print(f"\nAttempt {attempt+1}/{num_attempts}")
        total_params = num_layers * 2
        x0 = np.zeros(total_params, dtype=np.float64)
        for layer in range(num_layers):
            x0[layer*2:layer*2+2] = np.random.normal(-1, 1, 2)
        
        if target_level == 0:
            def loss_wrapper(p):
                return torch_wrapper_loss(p, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers)
            opt_method = 'L-BFGS-B'
            bounds = [(-3, 3)] * total_params
            jac = lambda p: loss_wrapper(p)[1]
            func = lambda p: loss_wrapper(p)[0]
        else:
            def loss_wrapper(p):
                return torch_wrapper_penalty(p, H_tensor, lower_states_tensor, L, I2, Sz, Sp, Sm,
                                             perms, inv_perms, num_layers, penalty_weight)
            opt_method = 'L-BFGS-B'
            bounds = [(-3, 3)] * total_params
            jac = lambda p: loss_wrapper(p)[1]
            func = lambda p: loss_wrapper(p)[0]
        
        try:
            result = minimize(func, x0, method=opt_method, jac=jac,
                              options={'maxiter': maxiter, 'ftol': 1e-12, 'gtol': 1e-10},
                              bounds=bounds)
            final_params = result.x
            final_loss = result.fun
            
            params_list = []
            for layer in range(num_layers):
                re_part = final_params[layer*2]
                im_part = final_params[layer*2+1]
                params_list.append([re_part, im_part])
            psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)
            psi_numpy = psi.cpu().detach().numpy()
            norm = np.vdot(psi_numpy, psi_numpy)
            
            if sp.issparse(H):
                H_psi = H.dot(psi_numpy)
            else:
                H_psi = H @ psi_numpy
            energy = np.real(np.vdot(psi_numpy, H_psi) / norm)

            fidelity = 0.0
            for idx in target_indices:
                overlap = np.abs(np.vdot(psi_numpy, exact_states[idx]['state']))**2 / norm
                fidelity += overlap

            lower_overlap = 0.0
            for idx in lower_indices:
                lower_overlap += np.abs(np.vdot(psi_numpy, exact_states[idx]['state']))**2 / norm
            
            print(f"  Energy = {energy:.8f}, Fidelity (target) = {fidelity:.8f}, Lower overlap = {lower_overlap:.8e}")
            
            if energy < best_energy:
                best_energy = energy
                best_params = final_params
                best_psi = psi_numpy
                best_fidelity = fidelity
        except Exception as e:
            print(f"  Optimization failed: {e}")
            continue
    
    print(f"\n===== Final result for target level {target_level} =====")
    print(f"Best energy: {best_energy:.8f}")
    print(f"Fidelity with target subspace: {best_fidelity:.8f}")
    for idx in target_indices:
        overlap = np.abs(np.vdot(best_psi, exact_states[idx]['state']))**2 / np.vdot(best_psi, best_psi)
        print(f"  Overlap^2 with state {idx} (E={exact_states[idx]['energy']:.8f}): {overlap:.8f}")
    if lower_indices:
        lower_total = 0.0
        for idx in lower_indices:
            ov = np.abs(np.vdot(best_psi, exact_states[idx]['state']))**2 / np.vdot(best_psi, best_psi)
            lower_total += ov
        print(f"Total overlap with lower subspaces: {lower_total:.8e}")
    
    return best_energy, best_params, best_psi, exact_states, best_fidelity

if __name__ == "__main__":
    L = 6
    delta = 1.0
    lambda_val = 0.3  
    num_layers = L // 2
    num_attempts = 10
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    target_level = 1   
    use_fitted_ground = True   
    
    best_energy, best_params, best_psi, exact_states, fidelity = optimize_subspace(
        model="NextNearest",
        L=L,
        target_level=target_level,
        penalty_weight=20.0,
        num_layers=num_layers,
        num_attempts=num_attempts,
        maxiter=20000,
        device=device,
        use_fitted_ground=use_fitted_ground,
        delta=delta,
        lambda_val=lambda_val
    )
    
    if best_params is not None:
        print("\nOptimized Bethe parameters (u per layer):")
        for layer in range(num_layers):
            re = best_params[layer*2]
            im = best_params[layer*2+1]
            print(f"  Layer {layer+1}: {re:.6f} + {im:.6f}i")


===== Subspace optimization =====
Model: NextNearest, L=6, target level = 1
Parameters: {'delta': 1.0, 'lambda_val': 0.6}
Degeneracy groups (energy, multiplicity):
  Group 0: energy = -9.60000000, multiplicity = 1
  Group 1: energy = -8.66476152, multiplicity = 1
  Group 2: energy = -6.99332591, multiplicity = 3
  Group 3: energy = -6.57359245, multiplicity = 6
  Group 4: energy = -5.60000000, multiplicity = 2
  Group 5: energy = -4.00000000, multiplicity = 6
  Group 6: energy = -1.60000000, multiplicity = 1
Target subspace indices: [1]
Lower subspace indices: [0]
Using fitted Bethe ground state for orthogonality penalty (instead of exact ground).
  -> Fitting ground state with Bethe ansatz (energy minimization)...
  Fitted ground state energy = -9.54811933

Attempt 1/10
  Energy = -6.19432862, Fidelity (target) = 0.00227527+0.00000000j, Lower overlap = 3.99825673e-03+0.00000000e+00j

Attempt 2/10
  Energy = -8.21719661, Fidelity (target) = 0.95190507+0.00000000j, Lower overlap = 3.28